# AFT-010: Context Exhaustion -- Demo**Class:** Context Budget | **Severity:** P1Demonstrates how conversation history growth silently truncates the system promptin long-running sessions (OpenClaw/Ari WhatsApp pattern).

In [ ]:
import jsondef count_tokens(msgs):    return sum(len(json.dumps(m))//4 for m in msgs)def system_prompt():    return {"role":"system","content":(        "You are Ari, a bilingual WhatsApp assistant. CRITICAL RULES:\n"        "1. Always respond in the user's preferred language.\n"        "2. Remember dietary preferences across the conversation.\n"        "3. Never recommend restaurants conflicting with dietary restrictions.\n"        "4. Sign every message with '-- Ari'.")}

## Build a 50-turn conversation

In [ ]:
def simulate_convo(n):    msgs = [system_prompt()]    msgs.append({"role":"user","content":"Hola! Prefiero hablar en espanol."})    msgs.append({"role":"assistant","content":"Hola! Con gusto. -- Ari"})    msgs.append({"role":"user","content":"Soy vegetariana, recuerda esto."})    msgs.append({"role":"assistant","content":"Anotado, vegetariana. -- Ari"})    for i in range(3, n+1):        msgs.append({"role":"user","content":f"Pregunta {i}: recomienda cena zona {i}. "+("detalle "*20)})        msgs.append({"role":"assistant","content":f"Resp {i}: Restaurante Verdura {i}. "+("info "*25)+" -- Ari"})    return msgsconvo = simulate_convo(50)print(f"{len(convo)} messages, ~{count_tokens(convo)} tokens")

## The Failure: Silent API TruncationAPI drops messages from the front. System prompt goes first. No error.

In [ ]:
MAX = 8000def api_truncate(msgs, mx):    r = list(msgs)    while count_tokens(r) > mx and len(r) > 1:        r.pop(0)    return rtrunc = api_truncate(list(convo), MAX)has_sys = any(m["role"]=="system" for m in trunc)print(f"After truncation: {len(trunc)} msgs, system_prompt={has_sys}")print("Agent now: responds in English, recommends meat, no signature")

## Wrong Fix: Naive Front-TrimKeeps system prompt but loses early-session preferences.

In [ ]:
def naive_trim(msgs, mx):    r = list(msgs)    while count_tokens(r) > mx and len(r) > 2:        for i,m in enumerate(r):            if m["role"]!="system":                r.pop(i); break    return rnaive = naive_trim(list(convo), MAX)has_sys = any(m["role"]=="system" for m in naive)has_diet = any("vegetarian" in m.get("content","").lower() for m in naive)print(f"Naive trim: sys_prompt={has_sys}, diet_pref={has_diet}")print("System prompt survives but user preferences are gone.")

## Correct Fix: Proactive Summarization at 60%

In [ ]:
def smart_summarize(msgs, mx, threshold=0.6):    if count_tokens(msgs) < int(mx*threshold):        return msgs    sys = msgs[0]    recent = msgs[-6:]    summary = {"role":"assistant","content":        "[Summary: User prefers Spanish. User is vegetarian. "        f"{(len(msgs)-7)//2} restaurant recommendations discussed.]"}    return [sys, summary] + recentsmart = smart_summarize(list(convo), MAX)has_sys = any(m["role"]=="system" for m in smart)has_diet = any("vegetarian" in m.get("content","").lower() for m in smart)print(f"Smart summary: {len(smart)} msgs, ~{count_tokens(smart)} tokens")print(f"sys_prompt={has_sys}, diet_in_summary={has_diet}")print(f"Budget used: {count_tokens(smart)/MAX:.0%} -- plenty of headroom")

## Key TakeawayNo context management works at the point of exhaustion.Summarize at 60% -- not at the limit.**Detection:** `session_tokens / max_context > 0.55` without recent summarization.